In [ ]:
import pandas as pd

# Load the combined datasets
prices = pd.read_csv(
    "../data/processed/combined_farmgate_prices.csv",
    parse_dates=["Date"]
)

# Check dataset
print("Dataset shape:", prices.shape)
print("\nData types:")
print(prices.dtypes)
print("\nMissing prices:", prices["Price"].isna().sum())

In [ ]:
# Find records where the price is zero
zero_prices = prices[prices["Price"] == 0]

print("Total zero prices:", len(zero_prices))
display(zero_prices)

In [ ]:
# Create a copy to preserve the original dataset
cleaned_prices = prices.copy()

# Replace zero prices with NaN
cleaned_prices["Price"] = cleaned_prices["Price"].mask(
    cleaned_prices["Price"] == 0
)

print("Missing prices:", cleaned_prices["Price"].isna().sum())
print("Remaining zeros:", (cleaned_prices["Price"] == 0).sum())
print("Dataset shape:", cleaned_prices.shape)


In [ ]:
# Find the latest available price date for each commodity
last_dates = (
    cleaned_prices.dropna(subset=["Price"])
    .groupby(["Category", "Commodity"])["Date"]
    .max()
    .rename("Last_Observed_Date")
)

# Add the latest observed date to each record
price_check = cleaned_prices.join(
    last_dates, on=["Category", "Commodity"]
)

# Identify the two types of missing values
missing = price_check["Price"].isna()

historical = missing & (price_check["Date"] <= price_check["Last_Observed_Date"])
trailing = missing & (price_check["Date"] > price_check["Last_Observed_Date"])

print("Historical missing:", historical.sum())
print("Trailing missing:", trailing.sum())


In [ ]:
# Display missing prices within historical records
historical_gaps = price_check.loc[
    historical, ["Category", "Commodity", "Date"]
].sort_values(["Category", "Commodity", "Date"])

display(historical_gaps)

print("Affected commodities:", historical_gaps["Commodity"].nunique())


In [ ]:
# Sort records by commodity and date
price_checks = price_check.sort_values(
    ["Category", "Commodity", "Date"]
).copy()

# Get nearby observed prices without changing the original values
groups = price_checks.groupby(["Category", "Commodity"])["Price"]

price_checks["Previous_Price"] = groups.ffill()
price_checks["Next_Price"] = groups.bfill()

# Display historical gaps and surrounding prices
display(price_checks.loc[
    price_checks["Price"].isna() &
    (price_checks["Date"] <= price_checks["Last_Observed_Date"]),
    ["Category" ,"Commodity", "Date", "Previous_Price", "Next_Price"]
])

In [ ]:
# Remove trailing missing months
# Keep records up to each commodity's last reported price
historical_data = price_check.loc[
    price_check["Date"] <= price_check["Last_Observed_Date"],
    ["Geolocation", "Category", "Commodity", "Date", "Price"]
].copy()

print("Remaining rows:", len(historical_data))
print("Remaining missing prices:", historical_data["Price"].isna().sum())


In [ ]:
# Create a copy of our historical dataset
cleaned_data = historical_data.sort_values(
    ["Category", "Commodity", "Date"]
).copy()

# Preserve original PSA prices
cleaned_data["Original_Price"] = cleaned_data["Price"]

# Fill short historical gaps within each commodity
cleaned_data["Price"] = (
    cleaned_data.groupby(["Category", "Commodity"])["Price"]
    .transform(lambda x: x.interpolate(
        method="linear", limit=5, limit_area="inside"
    ))
)

# Mark estimated prices
cleaned_data["Was_Interpolated"] = (
    cleaned_data["Original_Price"].isna() &
    cleaned_data["Price"].notna()
)

print("Interpolated prices:", cleaned_data["Was_Interpolated"].sum())
print("Remaining missing:", cleaned_data["Price"].isna().sum())
# cleaned_data[cleaned_data['Was_Interpolated']==True]


In [ ]:
cleaned_data

In [ ]:
# Validate the interpolated prices
# Display only the interpolated records
interpolated_records = cleaned_data.loc[
    cleaned_data["Was_Interpolated"],
    ["Category", "Commodity", "Date", "Original_Price", "Price"]
]

display(interpolated_records)

print("Interpolated records:", len(interpolated_records))
print("Invalid prices:", (cleaned_data["Price"] <= 0).sum())


In [ ]:
# Validate Original Prices
# Check whether any original PSA prices were changed
changed_prices = (
    cleaned_data["Original_Price"].notna() &
    (cleaned_data["Price"] != cleaned_data["Original_Price"])
)

print("Changed original prices:", changed_prices.sum())
print("Duplicate records:", cleaned_data.duplicated(
    subset=["Category", "Commodity", "Date"]
).sum())
print("Total records:", len(cleaned_data))


In [ ]:
# Save the Cleaned Dataset
# Sort the final dataset chronologically
cleaned_data = cleaned_data.sort_values(
    ["Category", "Commodity", "Date"]
)

# Select columns for the final CSV
final_data = cleaned_data[
    ["Geolocation", "Category", "Commodity", "Date", "Price", "Was_Interpolated"]
].copy()

# Save the cleaned dataset
final_data.to_csv(
    "../data/processed/cleaned_farmgate_prices.csv",
    index=False
)

print("Total records:", len(final_data))
print("Commodities:", final_data["Commodity"].nunique())
print("Missing prices:", final_data["Price"].isna().sum())
print("Interpolated prices:", final_data["Was_Interpolated"].sum())


In [ ]:
# Display all unique commodities by category
commodities = final_data[
    ["Category", "Commodity"]
].drop_duplicates().sort_values(["Category", "Commodity"])

display(commodities)

print("Total commodities:", len(commodities))
